In [1]:
# Input: A directed graph with weight matrix `weight' and a start vertex `s'.
# Output: An array `D' of distances as explained above.

graph = {
    'A': {'B': 6, 'C': 5, "D": 1},
    'B': {'A': 6, 'D': 5, "E": 3},
    'C': {'A': 5, 'D': 5, 'F': 2},
    'D': {'A': 1, 'B': 5, "C": 5, "E": 6, "F": 4},
    'E': {'B': 3, 'D': 6, "F": 6},
    'F': {'C': 2, 'D': 4, 'E': 4},
}

In [14]:
# Graph to mermaid
def to_mermaid(graph):
    mermaid = "```mermaid\nflowchart LR\n"

    for v, adj_list in graph.items():
        # `S`tating Node
        mermaid += " " * 4
        mermaid += f"{v}(({v}))\n"

        # adding Edges
        for u, w in adj_list.items():
            mermaid += " " * 4
            mermaid += f"{v}-- {w} -->{u}\n"

        mermaid += "\n"

    print(mermaid, "```")

to_mermaid(graph)

```mermaid
flowchart LR
    A((A))
    A-- 6 -->B
    A-- 5 -->C
    A-- 1 -->D

    B((B))
    B-- 6 -->A
    B-- 5 -->D
    B-- 3 -->E

    C((C))
    C-- 5 -->A
    C-- 5 -->D
    C-- 2 -->F

    D((D))
    D-- 1 -->A
    D-- 5 -->B
    D-- 5 -->C
    D-- 6 -->E
    D-- 4 -->F

    E((E))
    E-- 3 -->B
    E-- 6 -->D
    E-- 6 -->F

    F((F))
    F-- 2 -->C
    F-- 4 -->D
    F-- 4 -->E

 ```


```mermaid
flowchart LR
    A((A))
    A-- 6 ---B
    A-- 5 ---C
    A-- 1 ---D

    B((B))
    B-- 5 ---D
    B-- 3 ---E

    C((C))
    C-- 5 ---D
    C-- 2 ---F

    D((D))
    D-- 6 ---E
    D-- 4 ---F

    E((E))
    E-- 6 ---F

    F((F))
```

# Prim's Algorithm
Suppose that we already have a spanning tree connecting some set of vertices `S`. 
Then we can consider all the edges which connect a vertex in `S` to one outside of `S`, and add to `S` one of those that has minimal weight.
This cannot possibly create a circle, since it must add a vertex not yet in `S`. 

This process can be repeated, starting with any vertex to be the sole element of `S`, which is a trivial minimal spanning tree containing no edges. 

This approach is known as Prim's algorithm.

In [3]:
# Min-Priority Queue

class MinHeap:
    def __init__(self):
        self.heap = []

    # return the number of nodes in a heap
    def size(self):
            return len(self.heap)

    # check if the heap is empty
    def is_empty(self):
        return len(self.heap) == 0

    # string representation of a heap
    def __str__(self):
        return str(self.heap)

    # swap the heap values
    def swap(self, i, j):
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    # calculate the index of a node's parent
    def parent(self, index):
        return (index - 1) // 2

    # calculate the index of a node's left child
    def left_child(self, index):
        return 2 * index + 1

    # calculate the index of a node's right_child
    def right_child(self, index):
        return 2 * index + 2

    # check if a node at a given index has a parent
    def has_parent(self, index):
        return self.parent(index) >= 0

    # check if a node at a given index has a left child
    def has_left_child(self, index):
        return self.left_child(index) < len(self.heap)

    # check if a node at a given index has a right child
    def has_right_child(self, index):
        return self.right_child(index) < len(self.heap)
       
    # heapify 
    def heapify(self, array):
        self.heap = array
        for i in range(len(self.heap) - 1, -1, -1):
            if  self.has_left_child(i):
                self.heapify_down(i)
        
    # heapify down
    def heapify_down(self, index):
        if not self.has_left_child(index):
            return
        smaller_child_index = self.left_child(index)
        if self.has_right_child(index): 
            if self.heap[self.right_child(index)] < self.heap[smaller_child_index]:
                smaller_child_index = self.right_child(index)
        if self.heap[index] > self.heap[smaller_child_index]:
            self.swap(index, smaller_child_index)
        if self.has_left_child(smaller_child_index):
            self.heapify_down(smaller_child_index)
    
    # heapify up
    def heapify_up(self, index):
        if self.has_parent(index): 
            parent_index = self.parent(index)
            # compare the value to its parent and swap if necessary
            if self.heap[index] < self.heap[self.parent(index)]:
                parent_index = self.parent(index)
                self.swap(index, parent_index)
                # run the code for parent                 
                if self.has_parent(parent_index):
                    self.heapify_up(parent_index)

    # insert into heap
    def insert(self, value):
        self.heap.append(value)
        index = len(self.heap) - 1
        while self.has_parent(index) and self.heap[index] < self.heap[self.parent(index)]:
            parent_index = self.parent(index)
            self.swap(index, parent_index)
            index = parent_index
    
    # extract min from heap
    def extract_min(self):
        min_value = self.heap[0]
        if self.size() <= 1:
            self.heap = []
            return min_value
        self.heap = [self.heap[-1]] + self.heap[1:-1]
        index = 0
        while True:
            if not self.has_left_child(index):
                break
            smaller_child_index = self.left_child(index)
            if self.has_right_child(index): 
                if self.heap[self.right_child(index)] < self.heap[self.left_child(index)]:
                    smaller_child_index = self.right_child(index)
            if self.heap[index] > self.heap[smaller_child_index]:
                self.swap(index, smaller_child_index)
            else:
                break 
            index = smaller_child_index
        return min_value
    
    # peek
    def peek(self):
        return self.heap[0]
    

class PriorityQueue(MinHeap):
    def __init__(self):
        super().__init__()

    def insert(self, priority, value):
        return super().insert((priority, value))
    
    # change the priority (key) of an element already in the priority queue
    def change_key(self, value, new_priority):
        # search for value
        for i in range(self.size()):
            # if value is found
            if self.heap[i][1] == value:
                # change priority
                old_priority = self.heap[i][0]
                self.heap[i] = (new_priority, value)
                # heapify after changing priority to maintain heap property.
                # we can simply call heapify but heapify_up/down is more efficient
                if new_priority < old_priority:
                    self.heapify_up(i)
                else:
                    self.heapify_down(i)
                return


In [15]:
# Prim's Algorithm

from pprint import pprint
from typing import Dict


def prim(graph: Dict[str, Dict[str, int]]) -> Dict[str, Dict[str, int]]:
    unvisited_nodes = set(graph.keys())
    priority_queue = PriorityQueue()

    # Adding arbitrary node
    start_node = unvisited_nodes.pop()
    for u, w in graph[start_node].items():
        priority_queue.insert(w, (start_node, u))

    spanning_tree = {start_node: {}}
    total = 0

    while not priority_queue.is_empty():
        w, edge = priority_queue.extract_min()
        v, u = edge

        # Check whether a new edge doesn't make a loop
        if u in unvisited_nodes:
            print(f"Edge {edge} doesn't make a loop.")
            unvisited_nodes.discard(u)
            spanning_tree[v][u] = w
            total += w
            spanning_tree[u] = {}

            # Adding edges from u
            for node, weight in graph[u].items():
                if node in unvisited_nodes:
                    priority_queue.insert(weight, (u, node))
        
        else:
            print(f"Edge {edge} makes a loop.")


    print(f"Total weight: {total}")
    return spanning_tree


st = prim(graph)
pprint(st)
print()

to_mermaid(st)



Edge ('A', 'D') doesn't make a loop.
Edge ('D', 'F') doesn't make a loop.
Edge ('F', 'C') doesn't make a loop.
Edge ('F', 'E') doesn't make a loop.
Edge ('E', 'B') doesn't make a loop.
Edge ('A', 'C') makes a loop.
Edge ('D', 'B') makes a loop.
Edge ('D', 'C') makes a loop.
Edge ('A', 'B') makes a loop.
Edge ('D', 'E') makes a loop.
Total weight: 14
{'A': {'D': 1},
 'B': {},
 'C': {},
 'D': {'F': 4},
 'E': {'B': 3},
 'F': {'C': 2, 'E': 4}}

```mermaid
flowchart LR
    A((A))
    A-- 1 -->D

    D((D))
    D-- 4 -->F

    F((F))
    F-- 2 -->C
    F-- 4 -->E

    C((C))

    E((E))
    E-- 3 -->B

    B((B))

 ```


## Spanning tree (result)
```mermaid
flowchart LR
    A((A))
    A-- 1 -->D

    D((D))
    D-- 4 -->F

    F((F))
    F-- 2 -->C
    F-- 4 -->E

    C((C))

    E((E))
    E-- 3 -->B

    B((B))

 ```